In [1]:
# Imports and setup
import json
import warnings
import torch
from pprint import pprint


import pandas as pd
from sklearn.preprocessing import LabelEncoder

from benchmark.config import DATA_PROCESSED, N_TRIALS, NUM_FOLDS
from benchmark.models.xgboost_model import (
    build_pipeline_clf,
    build_preprocessor,
    evaluate_clf,
    export_results_clf,
    final_fit_clf,
    get_search_space,
    run_search_clf,
)

warnings.filterwarnings("ignore")

In [10]:
# Main config
data_name = "obesity_level"
processed = DATA_PROCESSED / data_name
k_folds = NUM_FOLDS
n_trials = N_TRIALS

xgb_device = "cuda" if torch.cuda.is_available() else "cpu"
# xgb_device = "cpu"
print("XGBoost device:", xgb_device)

XGBoost device: cuda


In [3]:
# Load data and metadata
train_df = pd.read_csv(processed / "train.csv")
test_df = pd.read_csv(processed / "test.csv")

with open(processed / "metadata.json", "r") as file:
    metadata = json.load(file)

print("Dataset path:", processed)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nMetadata:")
pprint(metadata)

Dataset path: /home/shado/college_project/comp9417/9417GroupProject/data/processed/obesity_level
Train shape: (1688, 17)
Test shape: (423, 17)

Metadata:
{'bin_cols': ['Gender',
              'family_history_with_overweight',
              'FAVC',
              'SMOKE',
              'SCC'],
 'cat_cols': ['CAEC', 'CALC', 'MTRANS'],
 'num_cols': ['TUE', 'FAF', 'CH2O', 'FCVC', 'Age', 'Height', 'Weight', 'NCP'],
 'target_col': 'NObeyesdad'}


In [4]:
# Split x and y
target_column = metadata["target_col"]

x_train_full = train_df.drop(columns=[target_column]).copy()
x_test = test_df.drop(columns=[target_column]).copy()

label_encoder = LabelEncoder()
y_train_full = label_encoder.fit_transform(train_df[target_column])
y_test = label_encoder.transform(test_df[target_column])

class_names = label_encoder.classes_
n_classes = len(class_names)

print("\nEncoded classes:")
print(dict(enumerate(class_names)))


Encoded classes:
{0: 'Insufficient_Weight', 1: 'Normal_Weight', 2: 'Obesity_Type_I', 3: 'Obesity_Type_II', 4: 'Obesity_Type_III', 5: 'Overweight_Level_I', 6: 'Overweight_Level_II'}


In [5]:
# Build preprocessor and run hyperparameter search
preprocessor = build_preprocessor(metadata)
pipeline = build_pipeline_clf(preprocessor, n_classes, device=xgb_device, n_jobs=1)
search_space = get_search_space()

best_search_params, best_model_params, best_cv_f1, search_time = run_search_clf(
    pipeline,
    search_space,
    x_train_full,
    y_train_full,
    n_classes=n_classes,
    k_folds=k_folds,
    n_trials=n_trials,
)

print("\nBest CV F1:", best_cv_f1)
print("\nBest hyperparameters:")
pprint(best_model_params)


Best CV F1: 0.9687992781301554

Best hyperparameters:
{'colsample_bytree': 0.9030187553597947,
 'gamma': 2.1073424781242878e-08,
 'learning_rate': 0.1561985292624422,
 'max_depth': 7,
 'min_child_weight': 1,
 'n_estimators': 265,
 'reg_alpha': 0.001332030529288859,
 'reg_lambda': 0.1006191384018849,
 'subsample': 0.8715208985892365}


In [6]:
# Final fit with best params
final_pipeline, fit_time = final_fit_clf(
    preprocessor,
    best_search_params,
    x_train_full,
    y_train_full,
    n_classes=n_classes,
    device=xgb_device,
)

In [7]:
# Evaluate on train and test
train_metrics = evaluate_clf(final_pipeline, x_train_full, y_train_full, n_classes)
test_metrics = evaluate_clf(final_pipeline, x_test, y_test, n_classes)

In [8]:
# Final results
print("\nFinal results:")

print("Search time (seconds):", search_time)
print("Final training time (seconds):", fit_time)
print("Best CV F1:", best_cv_f1)

print("\nTrain Accuracy:", train_metrics["accuracy"])
print("Train AUC-ROC:", train_metrics["auc_roc"])
print("Train F1:", train_metrics["f1"])
print("Train Precision:", train_metrics["precision"])
print("Train Recall:", train_metrics["recall"])
print(
    "Train inference time per sample (seconds):",
    train_metrics["infer_time_per_sample_s"],
)

print("\nTest Accuracy:", test_metrics["accuracy"])
print("Test AUC-ROC:", test_metrics["auc_roc"])
print("Test F1:", test_metrics["f1"])
print("Test Precision:", test_metrics["precision"])
print("Test Recall:", test_metrics["recall"])
print(
    "Test inference time per sample (seconds):", test_metrics["infer_time_per_sample_s"]
)


Final results:
Search time (seconds): 42.63172700299765
Final training time (seconds): 0.6879551490055746
Best CV F1: 0.9687992781301554

Train Accuracy: 1.0
Train AUC-ROC: 1.0
Train F1: 1.0
Train Precision: 1.0
Train Recall: 1.0
Train inference time per sample (seconds): 1.3638370262200434e-05

Test Accuracy: 0.966903073286052
Test AUC-ROC: 0.9990346395487394
Test F1: 0.9661313471105407
Test Precision: 0.9665026982467569
Test Recall: 0.9658391325879011
Test inference time per sample (seconds): 6.673717492889728e-05


In [9]:
# Export model and results
export_results_clf(
    data_name,
    final_pipeline,
    best_model_params,
    train_metrics,
    test_metrics,
    fit_time,
    search_time,
    best_cv_f1,
    y_test,
    class_names,
)


Model saved to /home/shado/college_project/comp9417/9417GroupProject/models/obesity_level/xgboost.joblib
Pipeline saved to /home/shado/college_project/comp9417/9417GroupProject/models/obesity_level/xgboost_pipeline.joblib
Results saved to /home/shado/college_project/comp9417/9417GroupProject/results/obesity_level/xgboost.json
